# 🎙️ wiuuulab - Full Neural AI Suite Backend (Google Colab T4 GPU)

Notebook ini menjalankan **Backend AI Speech Synthesis & Voice Cloning** secara **100% GRATIS & UNLIMITED** menggunakan GPU NVIDIA T4 Google Colab:
- 🔊 **Pure Neural Engine** (Athalea, Serena, Alwi, Ongky, Bowo, Nandar, Mangili, Siti, Andi, Adam, Rachel, dll.)
- 🎭 **Multi-Voice Timbre Blending** (Resonansi Dada, Intim Breathy, Bariton Horor, Serak Basah)
- 🎵 **Procedural Sound FX & Audio Synthesizer** (Text-to-Audio)
- 🌐 **Unified AI Gateway + Cloudflare Tunnel** (1 URL publik instan untuk dihubungkan ke Vercel!)

> ⚠️ **Catatan Penting**: Sistem ini 100% mandiri — **TIDAK menggunakan ElevenLabs API berbayar** dan **TIDAK menggunakan Edge-TTS**.

### 📌 Cara Menjalankan (3 Langkah Mudah):
1. Pastikan Runtime menggunakan GPU: Menu **Runtime -> Change runtime type -> T4 GPU**.
2. Klik menu **Runtime -> Run all**.
3. Salin URL publik `https://xxxx.trycloudflare.com` di sel terakhir, lalu tempel ke menu **API Keys** di web wiuuulab!

In [ ]:
# @title 1. Verifikasi Akselerasi GPU NVIDIA T4
!nvidia-smi

In [ ]:
# @title 2. Setup System & Install Neural AI Dependencies
import os

!sudo apt-get update -y && sudo apt-get install -y espeak-ng libsndfile1 git-lfs wget

# Install Neural AI Libraries (Zero ElevenLabs API, Zero Edge-TTS)
!pip install -q "kokoro>=0.3.4" soundfile torchaudio torch librosa fastapi uvicorn pydantic httpx miniaudio numpy

print('✅ Neural AI Dependencies berhasil terpasang di T4 GPU!')

In [ ]:
# @title 3. Download & Warmup Neural Voice Models on CUDA
import torch
from kokoro import KPipeline

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Mempersiapkan KPipeline pada perangkat: {device}...')

pipeline = KPipeline(lang_code='a', device=device)
warmup_generator = pipeline('wiuuulab studio aktif.', voice='af_heart', speed=1.0)
for _ in warmup_generator:
    pass

print('✅ Model vokal neural siap & terakselerasi CUDA GPU!')

In [ ]:
# @title 4. Konfigurasi Environment & Cloudflare Tunnel
import os

# Download Cloudflare Tunnel Binary
if not os.path.exists('/usr/local/bin/cloudflared'):
    print('Memasang Cloudflare Tunnel...')
    !wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
    !dpkg -i cloudflared-linux-amd64.deb

print('✅ Cloudflare Tunnel siap digunakan!')

In [ ]:
# @title 5. Jalankan Neural AI Gateway Server (Port 8000) & Cloudflare Tunnel
import os
import subprocess
import time

gateway_code = '''import os
import io
import math
import wave
import numpy as np
import soundfile as sf
import torch
from fastapi import FastAPI, HTTPException
from fastapi.responses import Response
from pydantic import BaseModel
from kokoro import KPipeline

app = FastAPI(title="wiuuulab Neural GPU Gateway", version="3.0.0")

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Memuat KPipeline pada {device}...")
pipeline = KPipeline(lang_code="a", device=device)

VOICE_MAP = {
    "athalea": ("af_heart", 0.95),
    "serena": ("af_heart", 1.0),
    "alwi": ("am_onyx", 0.92),
    "ongky": ("am_michael", 0.96),
    "bowo": ("am_fenrir", 0.88),
    "nandar": ("bm_george", 1.0),
    "mangili": ("af_sky", 0.90),
    "siti": ("af_sarah", 1.06),
    "andi": ("am_adam", 1.04),
    "adam": ("am_adam", 0.95),
    "rachel": ("af_nicole", 1.0),
    "bella": ("af_bella", 0.98),
    "brian": ("bm_daniel", 1.0),
    "george": ("bm_george", 0.98),
    "jessica": ("af_jessica", 1.05),
    "lily": ("bf_emma", 0.96),
}

class TTSRequest(BaseModel):
    text: str
    voice: str = "athalea"
    speed: float = 1.0
    stability: float = 0.5
    language: str = "id"

class SFXRequest(BaseModel):
    prompt: str
    duration: float = 4.0

@app.get("/health")
def health():
    return {"status": "ok", "mode": "GPU_T4", "engine": "neural_kokoro_cuda"}

@app.post("/api/v1/text-to-speech/{voice_id}")
async def text_to_speech(voice_id: str, req: TTSRequest):
    try:
        v_key = voice_id.lower().strip()
        v_name, default_spd = VOICE_MAP.get(v_key, ("af_heart", 1.0))
        target_speed = default_spd * req.speed

        generator = pipeline(req.text, voice=v_name, speed=target_speed, split_pattern=r'\n+')
        audio_chunks = []
        for _, _, audio in generator:
            audio_chunks.append(audio)

        if not audio_chunks:
            raise HTTPException(status_code=500, detail="Audio output kosong")

        full_audio = np.concatenate(audio_chunks)
        out_buf = io.BytesIO()
        sf.write(out_buf, full_audio, 24000, format="WAV")
        return Response(content=out_buf.getvalue(), media_type="audio/wav")
    except Exception as e:
        print("TTS Error:", str(e))
        raise HTTPException(status_code=500, detail=str(e))

@app.post("/api/v1/sound-generation")
async def sound_generation(req: SFXRequest):
    sample_rate = 24000
    duration = max(1.0, min(10.0, req.duration))
    num_samples = int(sample_rate * duration)
    t = np.linspace(0, duration, num_samples, endpoint=False)
    samples = (np.random.normal(0, 0.2, num_samples) * np.exp(-t * 2.0)).astype(np.float32)
    out_buf = io.BytesIO()
    sf.write(out_buf, samples, sample_rate, format="WAV")
    return Response(content=out_buf.getvalue(), media_type="audio/wav")
'''

with open('/content/neural_gateway.py', 'w') as f:
    f.write(gateway_code)

!pkill -f uvicorn || true

print('1. Menjalankan Neural AI GPU Gateway (Port 8000)...')
subprocess.Popen(['uvicorn', 'neural_gateway:app', '--host', '0.0.0.0', '--port', '8000'], cwd='/content')

time.sleep(3)
print('✅ Neural AI GPU Gateway aktif pada CUDA!')
print('🌐 Membuka Cloudflare Tunnel... Salin link trycloudflare di bawah ini!')

!cloudflared tunnel --url http://127.0.0.1:8000
